In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds

In [3]:
import os
print(os.getcwd())

c:\Users\augoa\OneDrive\Документы\DL


In [4]:
import keras

In [ ]:
import pathlib
import tensorflow as tf
from tensorflow import keras

dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir_raw = keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)

data_dir = pathlib.Path(data_dir_raw)
if (data_dir / "flower_photos").exists():
    data_dir = data_dir / "flower_photos"

print("Итоговый путь к датасету:", data_dir)

import os
subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
print("Найденные подпапки (классы):", subfolders)



Итоговый путь к датасету: C:\Users\augoa\.keras\datasets\flower_photos\flower_photos
Найденные подпапки (классы): ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


NameError: name 'SEED' is not defined

In [6]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

val_and_test_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

num_val_batches = val_and_test_ds.cardinality() // 2
val_ds = val_and_test_ds.take(num_val_batches)
test_ds = val_and_test_ds.skip(num_val_batches)

class_names = train_ds.class_names
print("Классы:", class_names)

Found 3670 files belonging to 5 classes.
Using 2569 files for training.
Found 3670 files belonging to 5 classes.
Using 1101 files for validation.
Классы: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [7]:

data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.1),
        keras.layers.RandomContrast(0.1),
        keras.layers.RandomTranslation(0.1, 0.1),
        keras.layers.RandomCrop(224, 224),
        keras.layers.RandomBrightness(0.1),
        keras.layers.RandomHeight(0.1),
        keras.layers.RandomWidth(0.1)
    ]
)
def augment(image, label):
    image = data_augmentation(image, training=True)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE).cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)



In [8]:
base_efB0 = keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_efB0.trainable = False

In [9]:
inputs = keras.Input(shape=(224, 224, 3))
x = keras.applications.efficientnet.preprocess_input(inputs)
x = base_efB0(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(len(class_names), activation='softmax')(x)

In [10]:
model_efB0 = keras.Model(inputs, outputs)

model_efB0.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
params_efB0 = sum([tf.keras.backend.count_params(w) for w in model_efB0.trainable_weights])
print(f"Обучаемых параметров (EfficientNetB0): {params_efB0:,}")

history_efB0 = model_efB0.fit(train_ds, validation_data=val_ds, epochs=5)

test_loss_efB0, test_acc_efB0 = model_efB0.evaluate(test_ds)
val_acc_efB0 = history_efB0.history['val_accuracy'][-1]
print(f"EfficientNetB0 -> Val Acc: {val_acc_efB0:.2%}, Test Acc: {test_acc_efB0:.2%}")

Обучаемых параметров (EfficientNetB0): 6,405
Epoch 1/5


c:\Users\augoa\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


81/81 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - accuracy: 0.6625 - loss: 0.9079 - val_accuracy: 0.8529 - val_loss: 0.4872
Epoch 2/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 76s 946ms/step - accuracy: 0.8408 - loss: 0.4857 - val_accuracy: 0.8897 - val_loss: 0.3660
Epoch 3/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 75s 926ms/step - accuracy: 0.8731 - loss: 0.3898 - val_accuracy: 0.8971 - val_loss: 0.3198
Epoch 4/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 75s 932ms/step - accuracy: 0.8968 - loss: 0.3302 - val_accuracy: 0.9081 - val_loss: 0.2964
Epoch 5/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 74s 914ms/step - accuracy: 0.9058 - loss: 0.2952 - val_accuracy: 0.9026 - val_loss: 0.2804
18/18 ━━━━━━━━━━━━━━━━━━━━ 15s 804ms/step - accuracy: 0.9174 - loss: 0.2750
EfficientNetB0 -> Val Acc: 90.26%, Test Acc: 91.74%


In [12]:
base_eff_ft = keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_eff_ft.trainable = True

for layer in base_eff_ft.layers[:-30]:
    layer.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = keras.applications.efficientnet.preprocess_input(inputs)
x = base_eff_ft(x, training=True)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(len(class_names), activation='softmax')(x)

model_eff_ft = keras.Model(inputs, outputs)

model_eff_ft.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

params_eff_ft = sum([tf.keras.backend.count_params(w) for w in model_eff_ft.trainable_weights])
print(f"Обучаемых параметров (EfficientNetB0 FT): {params_eff_ft:,}")

history_eff_ft = model_eff_ft.fit(train_ds, validation_data=val_ds, epochs=5)

test_loss_eff_ft, test_acc_eff_ft = model_eff_ft.evaluate(test_ds)
val_acc_eff_ft = history_eff_ft.history['val_accuracy'][-1]
print(f"EfficientNetB0 FT -> Val Acc: {val_acc_eff_ft:.2%}, Test Acc: {test_acc_eff_ft:.2%}")

Обучаемых параметров (EfficientNetB0 FT): 1,502,565
Epoch 1/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.2487 - loss: 1.6412 - val_accuracy: 0.3511 - val_loss: 1.5051
Epoch 2/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.4247 - loss: 1.4163 - val_accuracy: 0.5515 - val_loss: 1.2544
Epoch 3/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.5527 - loss: 1.2541 - val_accuracy: 0.6765 - val_loss: 1.0641
Epoch 4/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.6493 - loss: 1.1022 - val_accuracy: 0.7647 - val_loss: 0.9182
Epoch 5/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.7244 - loss: 0.9810 - val_accuracy: 0.8033 - val_loss: 0.8040
18/18 ━━━━━━━━━━━━━━━━━━━━ 15s 848ms/step - accuracy: 0.7666 - loss: 0.8510
EfficientNetB0 FT -> Val Acc: 80.33%, Test Acc: 76.66%


In [ ]:
model_cnn = keras.Sequential([
    keras.layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    
    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
     
    keras.layers.GlobalAveragePooling2D(),
    
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(class_names), activation='softmax')
])

model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

params_cnn = sum([tf.keras.backend.count_params(w) for w in model_cnn.trainable_weights])
print(f"Обучаемых параметров (Своя CNN): {params_cnn:,}")

history_cnn = model_cnn.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=10, 
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
)

test_loss_cnn, test_acc_cnn = model_cnn.evaluate(test_ds)
val_acc_cnn = history_cnn.history['val_accuracy'][-1]
print(f"Своя CNN -> Val Acc: {val_acc_cnn:.2%}, Test Acc: {test_acc_cnn:.2%}")

Обучаемых параметров (Своя CNN): 110,405
Epoch 1/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.3558 - loss: 1.4469 - val_accuracy: 0.4210 - val_loss: 1.2112
Epoch 2/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.4550 - loss: 1.2278 - val_accuracy: 0.4467 - val_loss: 1.1631
Epoch 3/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - accuracy: 0.4940 - loss: 1.1567 - val_accuracy: 0.4871 - val_loss: 1.1320
Epoch 4/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.5255 - loss: 1.1138 - val_accuracy: 0.5074 - val_loss: 1.1372
Epoch 5/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.5485 - loss: 1.0915 - val_accuracy: 0.5294 - val_loss: 1.0968
Epoch 6/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - accuracy: 0.5730 - loss: 1.0574 - val_accuracy: 0.5368 - val_loss: 1.1191
Epoch 7/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - accuracy: 0.5874 - loss: 1.0145 - val_accuracy: 0.5478 - val_loss: 1.1279
Epoch 8/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.6002 